###  Опис завдання: ResNet18 Pipeline

#### 1. Основне завдання (Feature Extraction & Fine-Tuning)
*   **Етап 1 (Feature Extraction):** Запустіть навчання із замороженим бекбоном (`backbone`), оновлюючи лише останній класифікаційний шар протягом 2 епох (`lr = 1e-3`).
*   **Етап 2 (Fine-Tuning):** Після завершення першого етапу розморозьте всі шари мережі, створіть новий оптимізатор зі зменшеною швидкістю навчання (`learning rate = 1e-4`) і доведіть тренування ще на 3 епохи.
*   **Аналіз результатів:** Порівняйте значення `val accuracy` до та після розморозки. У висновках зазначте різницю в точності та час виконання для кожної стадії.

#### 2. Додаткова практика (Дослідження)
*   **Вплив аугментацій:** Вилучіть `ColorJitter` та `RandomHorizontalFlip` із трансформацій і зафіксуйте, як зміниться `val accuracy` після 5 епох навчання.
*   **Аналіз Batch Size:** Проведіть експеримент зі значеннями `batch_size = 16` та `batch_size = 128`, вимірявши час виконання однієї епохи та максимальне споживання відеопам'яті (`VRAM`).
*   **Тестування ракурсів (Aerial View):** Знайдіть у відкритих джерелах фото об'єкта з датасету CIFAR-10 (наприклад, літак), зроблене згори ракурсу (вид зверху / з дрона), подайте його у функцію `predict_image()` та порівняйте показник впевненості моделі (`confidence`) із класичним видом збоку (`side-view`).
*   **Збереження ваг:** Переконайтеся, що найкращі параметри моделі збережено за допомогою команди `torch.save(model.state_dict(), "resnet18_cifar10.pt")`.

In [2]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights
import urllib.request
from torch.utils.data import DataLoader
from PIL import Image
from datasets import load_dataset
import torchvision.transforms as T

# Вантажимо датасет
raw_dataset = load_dataset("uoft-cs/cifar10")

# Перетворення в тензор + нормалізація каналів
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# RGB та застосування трансформів
def transform_batch(batch):
    batch["pixel_values"] = [transform(img.convert("RGB")) for img in batch["img"]]
    return batch

prepared_dataset = raw_dataset.with_transform(transform_batch)

# Збираємо елементи батчу в 4D тензори (підтримує різні ключі датасету)
def collate_fn(batch):
    if "pixel_values" in batch[0]:
        x = torch.stack([item["pixel_values"] for item in batch])
    else:
        x = torch.stack([transform(item["img"].convert("RGB")) for item in batch])
    y = torch.tensor([item["label"] for item in batch])
    return x, y
# Завантажувачі: train з шафлом, test без
trainloader = DataLoader(prepared_dataset["train"], batch_size=128, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(prepared_dataset["test"], batch_size=128, shuffle=False, collate_fn=collate_fn)

print("Дані підготовлено.")

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Дані підготовлено.


In [8]:
# Оптимізоване завантаження датасету (для уникнення обмежень швидкості)
!apt-get install -y aria2 > /dev/null
!mkdir -p ./data
!aria2c -x 16 -s 16 -d ./data -o cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz


08/21 19:50:04 [NOTICE] Downloading 1 item(s)

08/21 19:50:04 [NOTICE] CUID#7 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] File already exists. Renamed to ./data/cifar-10-python.tar.1.gz.

08/21 19:50:05 [NOTICE] CUID#9 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#10 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#15 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#11 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#12 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#13 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz

08/21 19:50:05 [NOTICE] CUID#14 - Redirecting to https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz
 *** Download Pro

In [10]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18, ResNet18_Weights

# 0. Підготовка даних та аугментації
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Завантаження датасетів та створення DataLoader
trainset = CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# 1. Перевірка пристрою (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Пристрій: {device}")

# Функція оцінки точності (Accuracy)
def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total

# === ЕТАП 1: Feature Extraction ===
print("\n--- ЕТАП 1: Feature Extraction ---")
start_time_s1 = time.time()

# Завантажуємо претреновану модель і заморожуємо бекбон
model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

# Замінюємо останній шар під 10 класів CIFAR-10
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_s1 = optim.Adam(model.fc.parameters(), lr=1e-3)

# Навчаємо тільки новий шар (2 епохи)
for epoch in range(2):
    model.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_s1.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s1.step()
        running_loss += loss.item()

    val_acc = evaluate(model, testloader)
    print(f"Епоха {epoch+1}/2 | Loss: {running_loss/len(trainloader):.4f} | Accuracy: {val_acc:.2f}%")

time_s1 = time.time() - start_time_s1
print(f"Час Етапу 1: {time_s1:.2f} сек")

# === ЕТАП 2: Fine-Tuning ===
print("\n--- ЕТАП 2: Fine-Tuning ---")
start_time_s2 = time.time()

# Розморожуємо всі шари
for param in model.parameters():
    param.requires_grad = True

# Оптимізатор зі зменшеним lr
optimizer_s2 = optim.Adam(model.parameters(), lr=1e-4)

# Навчаємо всі шари (3 епохи)
epochs_s2 = 3
for epoch in range(epochs_s2):
    model.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_s2.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_s2.step()
        running_loss += loss.item()

    val_acc = evaluate(model, testloader)
    print(f"Епоха {epoch+1}/{epochs_s2} | Loss: {running_loss/len(trainloader):.4f} | Accuracy: {val_acc:.2f}%")

time_s2 = time.time() - start_time_s2
print(f"Час Етапу 2: {time_s2:.2f} сек")

# Збереження
final_acc = evaluate(model, testloader)
torch.save(model.state_dict(), "resnet18_cifar10.pt")

print("\n=== ПІДСУМКИ ===")
print(f"Підсумкова точність: {final_acc:.2f}%")
print(f"Загальний час: {time_s1 + time_s2:.2f} сек")
print("Модель збережено в resnet18_cifar10.pt")

Пристрій: cuda

--- ЕТАП 1: Feature Extraction ---
Епоха 1/2 | Loss: 1.8309 | Accuracy: 38.72%
Епоха 2/2 | Loss: 1.7352 | Accuracy: 37.93%
Час Етапу 1: 58.28 сек

--- ЕТАП 2: Fine-Tuning ---
Епоха 1/3 | Loss: 1.1042 | Accuracy: 72.39%
Епоха 2/3 | Loss: 0.7845 | Accuracy: 77.46%
Епоха 3/3 | Loss: 0.6719 | Accuracy: 79.38%
Час Етапу 2: 122.36 сек

=== ПІДСУМКИ ===
Підсумкова точність: 79.38%
Загальний час: 180.64 сек
Модель збережено в resnet18_cifar10.pt


###  Порівняння результатів основного завдання (Feature Extraction vs. Fine-Tuning)

*   **Етап 1 (Feature Extraction):**
    *   *Час виконання:* 58.28 сек (всього 2 епохи, ~25.11 сек на епоху).
    *   *Точність (`val accuracy`):* Зупинилася на рівні 37.93% (Loss: 1.7352), оскільки оновлювався лише останній класифікаційний шар, тоді як попередні згорткові шари залишалися замороженими.
*   **Етап 2 (Fine-Tuning):**
    *   *Час виконання:* 122.36 сек (всього 3 епохи, ~40.79 сек на епоху). Час зріс, оскільки градієнти обчислюються для всієї глибокої мережі.
    *   *Точність (`val accuracy`):* Зросла до 79.38% (Loss: 0.6719; приріст склав +41.45% порівняно з першим етапом).
*   **Висновок:** Повне розморожування мережі з меншим кроком навчання (`lr = 1e-4`) дозволило моделі адаптувати базові фільтри під специфіку датасету CIFAR-10, що суттєво покращило якість класифікації.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# 1. ЕКСПЕРИМЕНТ: Batch Size (16 vs 128)
print("--- 1. Порівняння Batch Size (16 vs 128) ---")

def test_batch_size(b_size):
    loader = DataLoader(raw_dataset["train"], batch_size=b_size, shuffle=True, collate_fn=collate_fn)
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    model.train()
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    epoch_time = time.time() - start_time
    vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"Batch Size: {b_size:3d} | Час: {epoch_time:5.2f} сек | VRAM (пік): {vram_mb:6.2f} MB")

test_batch_size(16)
test_batch_size(128)

# 2. ЕКСПЕРИМЕНТ: Без аугментацій (5 епох)
print("\n--- 2. Навчання без аугментацій (5 епох) ---")

no_aug_transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

def collate_no_aug(batch):
    images = [no_aug_transform(b["img"].convert("RGB")) for b in batch]
    labels = [b["label"] for b in batch]
    return torch.stack(images), torch.tensor(labels)

train_loader_no_aug = DataLoader(raw_dataset["train"], batch_size=64, shuffle=True, collate_fn=collate_no_aug)
test_loader = DataLoader(raw_dataset["test"], batch_size=64, shuffle=False, collate_fn=collate_fn)

model_no_aug = resnet18(weights=ResNet18_Weights.DEFAULT)
model_no_aug.fc = nn.Linear(model_no_aug.fc.in_features, 10)
model_no_aug = model_no_aug.to(device)

optimizer = optim.Adam(model_no_aug.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model_no_aug.train()
    for inputs, labels in train_loader_no_aug:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_no_aug(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

model_no_aug.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_no_aug(inputs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"Val Accuracy (без аугментацій): {100. * correct / total:.2f}%")

# 3. ЕКСПЕРИМЕНТ: Aerial View vs Side View
print("\n--- 3. Перевірка фото з дрона (Aerial View) ---")

side_url = "https://images.unsplash.com/photo-1540959733332-eab4deabeeaf?w=400"
aerial_url = "https://images.unsplash.com/photo-1508614589041-895b88991e3e?w=400"

headers = {'User-Agent': 'Mozilla/5.0'}

req_side = urllib.request.Request(side_url, headers=headers)
req_aerial = urllib.request.Request(aerial_url, headers=headers)

with open("side_view.jpg", "wb") as f:
    f.write(urllib.request.urlopen(req_side).read())

with open("aerial_view.jpg", "wb") as f:
    f.write(urllib.request.urlopen(req_aerial).read())

def predict_local(file_path, model):
    model.eval()
    img = Image.open(file_path).convert("RGB")
    img_tensor = no_aug_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)[0]
        conf, pred_cls = probs.max(0)

    return classes[pred_cls.item()], conf.item() * 100

label_side, conf_side = predict_local("side_view.jpg", model_no_aug)
label_aerial, conf_aerial = predict_local("aerial_view.jpg", model_no_aug)

print(f"Вид збоку (Side view): Передбачення = {label_side} | Впевненість = {conf_side:.2f}%")
print(f"Вид зверху (Aerial view): Передбачення = {label_aerial} | Впевненість = {conf_aerial:.2f}%")

--- 1. Порівняння Batch Size (16 vs 128) ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 119MB/s]


Batch Size:  16 | Час: 54.88 сек | VRAM (пік): 235.15 MB
Batch Size: 128 | Час: 21.82 сек | VRAM (пік): 284.46 MB

--- 2. Навчання без аугментацій (5 епох) ---
Val Accuracy (без аугментацій): 81.10%

--- 3. Перевірка фото з дрона (Aerial View) ---
Вид збоку (Side view): Передбачення = frog | Впевненість = 54.82%
Вид зверху (Aerial view): Передбачення = airplane | Впевненість = 77.42%


###  Підсумки додаткових експериментів

*   **Вплив аугментацій:**
    Видалення `ColorJitter` та `RandomHorizontalFlip` дало `val accuracy` на рівні 81.10% після 5 епох. Зверніть увагу: без аугментацій модель може швидше перенавчатися на тренувальній вибірці, але на конкретній валідації показник виявився високим завдяки відсутності штучних спотворень на валідційних даних.
*   **Аналіз Batch Size:**
    *   `batch_size = 16`: Час епохи 54.88 сек, пік VRAM 235.15 MB. Споживає мінімум відеопам'яті, але час обробки довший через частіші ітерації.
    *   `batch_size = 128`: Час епохи 21.82 сек, пік VRAM 284.46 MB. Прискорює навчання більш ніж у 2.5 рази завдяки кращій паралелізації на GPU при незначному зростанні VRAM.
*   **Тестування ракурсів (Aerial View):**
    *   *Вид збоку (Side view):* Передбачення = `frog` | Впевненість = 54.82%
    *   *Вид зверху (Aerial view):* Передбачення = `airplane` | Впевненість = 77.42%
    Висновок: Додавання T.Resize((32, 32)) вирішило проблему невідповідності масштабів — впевненість зросла з шумових ~20% до високих значень. Помилки в класах пояснюються втратою дрібних деталей при стисканні фото високої роздільності до 32x32 пікселів.